In [2]:
# Imports
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import duckdb
from config.settings import DUCKDB_FILE
from src.database.database_manager import DatabaseManager

# Initialize database manager
db = DatabaseManager()

## Database Schema Overview

List every table in `jobs.duckdb` along with its row count, then show each table's column-level schema (name, type, nullability).

In [3]:
# All tables in the database, with row counts
tables = db.query("SHOW TABLES")['name'].tolist()

table_summary = pd.DataFrame([
    {'table': t, 'row_count': db.query(f"SELECT COUNT(*) as n FROM {t}").iloc[0]['n']}
    for t in tables
]).sort_values('table').reset_index(drop=True)

table_summary

,table,row_count
0,jobs,1026079
1,market_trends,0
2,raw_jobs,0
3,raw_jobs_flat,1048585
4,role_statistics,0
5,salary_benchmarks,0
6,skills_demand,0


In [4]:
# Column-level schema for every table
for t in tables:
    print(f"\n=== {t} ===")
    schema = db.query(f"DESCRIBE {t}")[['column_name', 'column_type', 'null', 'key']]
    display(schema)


=== jobs ===


,column_name,column_type,null,key
0,job_id,VARCHAR,NO,PRI
1,title,VARCHAR,NO,None
2,company,VARCHAR,NO,None
3,sector,VARCHAR,YES,None
4,sub_sector,VARCHAR,YES,None
5,location,VARCHAR,YES,None
6,salary_min,INTEGER,YES,None
7,salary_max,INTEGER,YES,None
8,salary_currency,VARCHAR,YES,None
9,experience_level,VARCHAR,YES,None



=== market_trends ===


,column_name,column_type,null,key
0,trend_id,VARCHAR,NO,PRI
1,trend_date,DATE,YES,None
2,sector,VARCHAR,YES,None
3,role,VARCHAR,YES,None
4,posting_count,INTEGER,YES,None
5,avg_salary,INTEGER,YES,None
6,skill_demand,VARCHAR,YES,None
7,experience_level,VARCHAR,YES,None
8,created_at,TIMESTAMP,YES,None



=== raw_jobs ===


...

,column_name,column_type,null,key
0,role_name,VARCHAR,NO,PRI
1,sector,VARCHAR,YES,None
2,total_postings,INTEGER,YES,None
3,avg_salary,INTEGER,YES,None
4,min_salary,INTEGER,YES,None
5,max_salary,INTEGER,YES,None
6,top_skills,VARCHAR,YES,None
7,experience_level,VARCHAR,YES,None
8,avg_experience_years,FLOAT,YES,None
9,last_updated,TIMESTAMP,YES,None



=== salary_benchmarks ===


,column_name,column_type,null,key
0,benchmark_id,VARCHAR,NO,PRI
1,role,VARCHAR,YES,None
2,experience_level,VARCHAR,YES,None
3,sector,VARCHAR,YES,None
4,salary_p25,INTEGER,YES,None
5,salary_p50,INTEGER,YES,None
6,salary_p75,INTEGER,YES,None
7,salary_p90,INTEGER,YES,None
8,count_samples,INTEGER,YES,None
9,last_updated,TIMESTAMP,YES,None



=== skills_demand ===


,column_name,column_type,null,key
0,skill_id,VARCHAR,NO,PRI
1,skill_name,VARCHAR,NO,UNI
2,total_postings,INTEGER,YES,None
3,avg_salary,INTEGER,YES,None
4,sectors,VARCHAR,YES,None
5,experience_levels,VARCHAR,YES,None
6,trend_direction,VARCHAR,YES,None
7,last_updated,TIMESTAMP,YES,None


## Data Quality: Salary Outliers

`SALARY_MAX_THRESHOLD` (previously capping `salary_max` at 100,000 SGD) was removed from the cleaning pipeline so unusually high salaries are no longer silently dropped. That surfaces a real data-quality issue in `SGJobData.csv`: a small number of postings have `salary_max` values that are obviously not real monthly SGD salaries (e.g. in the millions), most likely data-entry or scraping errors.

The cells below quantify how many rows are affected and how much they distort `AVG(salary_max)`.

In [6]:
# Overall salary distribution stats
stats = db.query("""
    SELECT
        COUNT(*) as total_rows,
        MIN(salary_max) as min_salary_max,
        MAX(salary_max) as max_salary_max,
        AVG(salary_max) as avg_salary_max,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary_max) as median_salary_max,
        STDDEV(salary_max) as stddev_salary_max
    FROM jobs
""")
stats

,total_rows,min_salary_max,max_salary_max,avg_salary_max,median_salary_max,stddev_salary_max
0,1026079,1000,25330000,5827.124626,4500.0,50712.904899


In [7]:
# How many rows look like data errors, at a few thresholds?
threshold_counts = db.query("""
    SELECT
        COUNT(*) FILTER (WHERE salary_max > 20000) as over_20k,
        COUNT(*) FILTER (WHERE salary_max > 50000) as over_50k,
        COUNT(*) FILTER (WHERE salary_max > 100000) as over_100k,
        COUNT(*) FILTER (WHERE salary_max > 1000000) as over_1m
    FROM jobs
""")
threshold_counts

,over_20k,over_50k,over_100k,over_1m
0,9087,455,274,12


In [8]:
# The worst offenders, to eyeball what the "error" actually looks like
top_outliers = db.query("""
    SELECT job_id, title, company, salary_min, salary_max
    FROM jobs
    ORDER BY salary_max DESC
    LIMIT 20
""")
top_outliers

,job_id,title,company,salary_min,salary_max
0,job_360c37e359b8,Accounts Executive - GT,RK RECRUITMENT PTE. LTD.,2800,25330000
1,job_907c7632e893,Executive Secretary,SAFRAN LANDING SYSTEMS SERVICES SINGAPORE PTE....,14719,23712119
2,job_328ac6bae1ea,Language Teacher,ASCOTT INTERNATIONAL MANAGEMENT PTE LTD,324072,20862169
3,job_c3dc2ee136c6,"Sales Associate (Home Audio, Retail)",RK RECRUITMENT PTE. LTD.,262482,15531134
4,job_f27dcac969c8,sales and operations manager,THALES DIS (SINGAPORE) PTE. LTD.,164428,14420727
5,job_76dec58924b4,Social media content creator,MINDFLEX EDUCATION PTE. LTD.,260117,13798518
6,job_1989c588df50,Resident Physician,ASCEND INTERNATIONAL TRAINING PTE. LTD.,108872,10734314
7,job_5b437ee302ab,Clinic assistant,HILL GROVE MEDICAL PTE. LTD.,1500,10000000
8,job_b0c20c512328,Junior Project Manager (IT Infrastructure),RECRUIT EXPRESS PTE LTD,267303,7859259
9,job_a4a6e4f86746,Senior Manager - Operations,BOND CAPITAL GROUP PTE. LTD.,107908,6142101


In [9]:
# Quantify the distortion: AVG(salary_max) with vs. without the >100k outliers
impact = db.query("""
    SELECT
        (SELECT AVG(salary_max) FROM jobs) as avg_with_outliers,
        (SELECT AVG(salary_max) FROM jobs WHERE salary_max <= 100000) as avg_without_outliers,
        (SELECT COUNT(*) FROM jobs WHERE salary_max > 100000) as outlier_rows,
        (SELECT COUNT(*) FROM jobs) as total_rows
""")
impact['pct_rows_are_outliers'] = (impact['outlier_rows'] / impact['total_rows'] * 100).round(3)
impact['avg_inflated_by'] = (impact['avg_with_outliers'] - impact['avg_without_outliers']).round(2)
impact

,avg_with_outliers,avg_without_outliers,outlier_rows,total_rows,pct_rows_are_outliers,avg_inflated_by
0,5827.124626,5632.112399,274,1026079,0.027,195.01
